# Exploratory Data Analysis

## Load data

Data from https://www.kaggle.com/datasets/muhammadshahidazeem/customer-churn-dataset.

We renamed the "testing" data set to "inference", because we will use the "training" set for model training, cross-validation and testing. We will treat the  "inference" set as "unseen" data for model training and instead use it later for inference and monitoring including data/concept drift.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from churn_mlops.data import load_raw_data

pd.set_option('display.width', 200)

df_train = load_raw_data("customer_churn_dataset-training.csv")
df_infer = load_raw_data("customer_churn_dataset-inference.csv")

cols_num = df_train.columns[df_train.dtypes != 'category'].to_list()
cols_cat = df_train.columns[df_train.dtypes == 'category'].to_list()

## Summary of data sets

Observations:
* Our function `load_raw_data` ensures that both dataframes have the same column data types.
* There are no missing values in the data.
* Difference in distribution of numerical variables (e.g. different mean and median), most distinct for
    * `Age`
    * `Support Calls`
    * `Payment Delay`
    * `Total Spend`
* Comparatively higher churn incidence in training data.

In [ ]:
print(df_train.info())

In [ ]:
print(df_infer.info())

In [ ]:
print(df_train.describe())

In [ ]:
print(df_infer.describe())

In [ ]:
# verify if there is any missing data
num_nulls_train = df_train.isna().sum()
num_nulls_infer = df_infer.isna().sum()
num_nulls = pd.concat([num_nulls_train,num_nulls_infer], axis='columns')
num_nulls.columns = ['training','inference']
print(num_nulls)

## Correlation

Observations:
* Very different picture comparing correlations for "training" and "inference" data sets.
* Hardly any correlation between dependent variables for "inference" set, as opposed to "training" set.
* Different correlation pattern with target variable "Churn", indicating concept drift.

In [ ]:
corr_train = df_train.corr(numeric_only=True)
corr_infer = df_infer.corr(numeric_only=True)

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(
    corr_train,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    annot=True
)
plt.title("Training data: correlation of numeric variables")
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(
    corr_infer,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    annot=True,
)
plt.title("Inference data: correlation of numeric variables")
plt.show()

## Distribution of categorical variables

Observations:
* `Gender`:             Shift from "Male" to "Female"
* `Contract Length`:    Initially lower share of "Monthly" subscriptions
* `Subscription Type`:  Equal share of "Basic", "Standard" and "Premium"

In [ ]:
# compare shares of categorical variables
cat_share_train = df_train[cols_cat].value_counts(normalize=True).sort_index()
cat_share_infer = df_infer[cols_cat].value_counts(normalize=True).sort_index()
cat_share = pd.concat([cat_share_train,cat_share_infer], axis='columns')
cat_share.columns = ['training','inference']
print(cat_share.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(
    data=cat_share,
    x='training',
    y='inference',
    hue='Contract Length',
    size='Subscription Type',
    style='Gender',
    ax=ax
    )
ax.plot(
    [0.03, 0.08],
    [0.03, 0.08],
    color="black",
    linestyle="--",
    label="equality"
)
ax.legend(
    fontsize=10,
    title_fontsize=12,
    bbox_to_anchor=(1.0, 1),
    loc="upper left"
)
plt.tight_layout()
plt.show()

In [ ]:
cat_share_train = df_train["Gender"].value_counts(normalize=True).sort_index()
cat_share_infer = df_infer["Gender"].value_counts(normalize=True).sort_index()
cat_share = pd.concat([cat_share_train,cat_share_infer], axis='columns')
cat_share.columns = ['training','inference']

ax = cat_share.T.plot(
    kind="bar",
    stacked=True,
    figsize=(8, 4)
)
ax.set_ylabel("Anteil")
ax.legend(title="Gender", bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

print(cat_share.round(3))

In [ ]:
cat_share_train = df_train["Contract Length"].value_counts(normalize=True).sort_index()
cat_share_infer = df_infer["Contract Length"].value_counts(normalize=True).sort_index()
cat_share = pd.concat([cat_share_train,cat_share_infer], axis='columns')
cat_share.columns = ['training','inference']

ax = cat_share.T.plot(
    kind="bar",
    stacked=True,
    figsize=(8, 4)
)
ax.set_ylabel("Anteil")
ax.legend(title="Contract Length", bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

print(cat_share.round(3))

In [ ]:
cat_share_train = df_train["Subscription Type"].value_counts(normalize=True).sort_index()
cat_share_infer = df_infer["Subscription Type"].value_counts(normalize=True).sort_index()
cat_share = pd.concat([cat_share_train,cat_share_infer], axis='columns')
cat_share.columns = ['training','inference']

ax = cat_share.T.plot(
    kind="bar",
    stacked=True,
    figsize=(8, 4)
)
ax.set_ylabel("Anteil")
ax.legend(title="Subscription Type", bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

print(cat_share.round(3))

## Distribution of numerical variables

Observations:
* The distribution of numerical variables differs between both datasets.
* Our sub-sample based pairs-plot (differentiating by churn) reveals somewhat artifical relation/distribution patterns.

In [ ]:
sns.kdeplot(df_train["Age"], fill=True, alpha=0.3, label="Training")
sns.kdeplot(df_infer["Age"], fill=True, alpha=0.3, label="Inference")
plt.legend()
plt.show()

In [ ]:
sns.kdeplot(df_train["Tenure"], fill=True, alpha=0.3, label="Training")
sns.kdeplot(df_infer["Tenure"], fill=True, alpha=0.3, label="Inference")
plt.legend()
plt.show()

In [ ]:
sns.kdeplot(df_train["Usage Frequency"], fill=True, alpha=0.3, label="Training")
sns.kdeplot(df_infer["Usage Frequency"], fill=True, alpha=0.3, label="Inference")
plt.legend()
plt.show()

In [ ]:
sns.histplot(df_train["Support Calls"], bins=11, alpha=0.3, label="Training")
sns.histplot(df_infer["Support Calls"], bins=11, alpha=0.3, label="Inference")
plt.legend()
plt.show()

In [ ]:
sns.kdeplot(df_train["Payment Delay"], fill=True, alpha=0.3, label="Training")
sns.kdeplot(df_infer["Payment Delay"], fill=True, alpha=0.3, label="Inference")
plt.legend()
plt.show()

In [ ]:
sns.kdeplot(df_train["Total Spend"], fill=True, alpha=0.3, label="Training")
sns.kdeplot(df_infer["Total Spend"], fill=True, alpha=0.3, label="Inference")
plt.legend()
plt.show()

In [ ]:
sns.kdeplot(df_train["Last Interaction"], fill=True, alpha=0.3, label="Training")
sns.kdeplot(df_infer["Last Interaction"], fill=True, alpha=0.3, label="Inference")
plt.legend()
plt.show()

In [ ]:
train_sample = df_train.sample(n=1000, random_state=42)
sns.pairplot(
    train_sample,
    vars=cols_num[:-1],
    hue="Churn",
    plot_kws={"alpha": 0.2}
)
plt.show()

## Relationship between churn and other variables

Observations:
* Churn incidence of 100% for
    * `Age` above 50,
    * (number of)* `Support Calls` of 6 and more, 
    * `Payment Delay` of more than 20 days*, 
    * `Total Spend` of 500 or less,
    * `Contract Length = "Monthly"`.
* Relatively higher churn incidence for
    * lower `Age` (below 50),
    * `Tenure` of 5 or less and between 12 and 24 (months)*,
    * `Usage Frequency` below 10 (days per month)*,
    * (number of)* `Support Calls` between 3 and 5,
    * `Last Interactio` more than 15 days* ago,
    * `Gender = "Female"`.

* `*` Note that the unit of variables is a context based guess, as the data set is lacking exact variable definitions.

In [ ]:
for var in ['Age', 'Tenure', 'Usage Frequency', 'Support Calls', 'Payment Delay', 'Last Interaction', 'Gender', 'Subscription Type', 'Contract Length']:
    plt.figure(figsize=(12,5))
    df_train.groupby(var)["Churn"].mean().plot(kind="bar")
    plt.title(f"Churn Incidence by {var}")
    plt.ylim([0,1])
    plt.show()

In [ ]:
var = "Total Spend"
tmp = df_train.copy()
tmp[var + " Binned"] = pd.cut(df_train[var], bins=range(100,1001,100), include_lowest=True, precision=0)

plt.figure(figsize=(12,5))
tmp.groupby(var + " Binned")["Churn"].mean().plot(kind="bar")
plt.title(f"Churn Incidence by {var}")
plt.ylim([0,1])
plt.show()

## Customer ID

Eventually, we were wondering, if customer IDs in both data sets correspond to one and the same customer (maybe at a later point in time).

Conclusion: Comparing variables by `CustomerID`, we are certain, that this is not the case (e.g. ~ 50% different gender).

In [ ]:
# identify overlapping records by index (CustomerID)
df_inner = df_train.merge(df_infer, how='inner', left_index=True, right_index=True, suffixes=('_trn','_tst'))
print(df_inner.shape)

In [ ]:
# compare colum values for records with equal index (CustomerID)
[(col,(df_inner[col+'_trn']==df_inner[col+'_tst']).sum()) for col in df_train.columns]